# Zero-shot open-model LLM baseline

Runs two open models over 76 targets × 3 conditions (228 prompts per model) and writes every answer
to `results/` in this Drive folder **as soon as it is finished**. If Colab disconnects, run cells 1–3
again and then the model cell that was running: finished items are skipped.

**Runtime:** `Runtime → Change runtime type → T4 GPU`. About 1–1.5 h per model on a T4.

## 1. Mount Drive, check the GPU

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
BASE = '/content/drive/MyDrive/task_inference_llm_baseline'   # change if you uploaded elsewhere
RESULTS = BASE + '/results'
import os
for f in ('config.json', 'items.csv', 'llm_lib.py', 'prompt_system.md', 'packets'):
    assert os.path.exists(os.path.join(BASE, f)), f'missing {f}'
!nvidia-smi --query-gpu=name,memory.total --format=csv

## 2. Install

In [ ]:
%pip install -q -U transformers accelerate bitsandbytes

## 3. Load the bundle

In [ ]:
import sys, importlib, json, gc
sys.path.insert(0, BASE)
import llm_lib as L
importlib.reload(L)
cfg, items = L.load_bundle(BASE)
print(cfg['version'], '| models:', cfg['models'])
print(items.groupby(['group', 'condition']).size().unstack())

## 4. (Optional) smoke test — first model, 3 prompts, into `results_smoke/`
Confirms the model loads and answers parse. About 3–5 minutes including the download.

In [ ]:
m = L.HFModel(cfg['models'][0], cfg)
L.run_model(BASE, BASE + '/results_smoke', m, limit=3)
print(L.compile_results(BASE + '/results_smoke')[['instance_id', 'condition', 'parse_ok', 'plausible_task_set', 'most_likely_task']])
del m; gc.collect()
import torch; torch.cuda.empty_cache()

## 5. Model 1 (all 228 prompts)

In [ ]:
m = L.HFModel(cfg['models'][0], cfg)
L.run_model(BASE, RESULTS, m)
del m; gc.collect()
import torch; torch.cuda.empty_cache()

## 6. Model 2 (all 228 prompts)

In [ ]:
m = L.HFModel(cfg['models'][1], cfg)
L.run_model(BASE, RESULTS, m)
del m; gc.collect()
import torch; torch.cuda.empty_cache()

## 7. Compile and zip for download

In [ ]:
df = L.compile_results(RESULTS)
print(df.groupby(['model', 'condition']).parse_ok.mean().unstack())
import shutil
shutil.make_archive(BASE + '/llm_results', 'zip', RESULTS)
print('wrote', BASE + '/llm_results.zip')